# Products
1. Define Schema
2. Load CSV
3. Category - Fill Down
4. Product column - Split into "Product" and "Segment" columns
5. Price column - Split into "Currency" and "MSRP" columns
6. Write Delta table


## 1. Create Schema

In [1]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

# Define the schema for products.csv
products_schema = StructType([
    StructField("ProductID", IntegerType(), False),   # Not nullable (assuming ProductID is always present)
    StructField("Product", StringType(), True),      # Nullable text field
    StructField("Category", StringType(), True),     # Nullable text field
    StructField("ManufacturerID", IntegerType(), True), # Nullable integer field
    StructField("Price", StringType(), True)        # Stored as text (string)
])


StatementMeta(, 3d6960f3-be80-4952-a9e9-b6c828514258, 3, Finished, Available, Finished)

## 2. Load CSV

In [2]:
df = spark.read.format("csv") \
    .option("header", "true") \
    .schema(products_schema) \
    .load("Files/USSales/products.csv")  # Ensure no trailing backslash here

# Display the DataFrame
display(df)


StatementMeta(, 3d6960f3-be80-4952-a9e9-b6c828514258, 4, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, b292b452-e54e-4d31-871b-93722720fe80)

## 3. Split Product column

In [3]:
# Split the Product column into Product and Segment

from pyspark.sql.functions import split

# Split the "Product" column into "Product" and "Segment"
df = df.withColumn("Segment", split(df["Product"], "\|")[1]) \
       .withColumn("Product", split(df["Product"], "\|")[0])  # Overwrite "Product" column

# Display the updated DataFrame
display(df)


StatementMeta(, 3d6960f3-be80-4952-a9e9-b6c828514258, 5, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 0aac7421-cbec-4bdd-bcd0-a99c01cee6be)

## 4. Fill Down "Category" column

In [4]:
# Fill Down the Category column

from pyspark.sql.window import Window
from pyspark.sql.functions import last

# Define a window that orders rows naturally
window_spec = Window.orderBy("ProductID").rowsBetween(Window.unboundedPreceding, 0)

# Fill down the Category column
df = df.withColumn("Category", last("Category", ignorenulls=True).over(window_spec))

# Display the updated DataFrame
display(df)


StatementMeta(, 3d6960f3-be80-4952-a9e9-b6c828514258, 6, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 98d644dc-a6c7-49db-826f-c5a152a439db)

## 5. Split Price column

In [5]:
# Split the Price column into "Currency" and "MSRP"

from pyspark.sql.functions import split, regexp_replace
from pyspark.sql.types import DecimalType

# Split the "Price" column into Currency and MSRP
df = df.withColumn("Currency", split(df["Price"], " ")[0]) \
       .withColumn("MSRP", regexp_replace(split(df["Price"], " ")[1], ",", "").cast(DecimalType(10,2))) \
       .drop("Price")  # Remove the original "Price" column

# Display the updated DataFrame
display(df)


StatementMeta(, 3d6960f3-be80-4952-a9e9-b6c828514258, 7, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, faf9d7fd-aa26-4e82-b36c-5b19a3622f16)

## 6. Write Delta table

In [6]:
# Let's write the Delta table

df.write.format("delta").mode("overwrite").saveAsTable("Products")

StatementMeta(, 3d6960f3-be80-4952-a9e9-b6c828514258, 8, Finished, Available, Finished)